# Shapes3D → five-block ViT-Tiny → the repository's SAE tracking pipeline

Train a small classifier on four Shapes3D shape classes, export its representations,
fit independent SAEs, run the repository's combined evaluator, and aggregate its saved
evolution patterns. The default backbone uses **five blocks, width 192, and three heads**.
All computation is on CPU; the configuration cell controls data size, width, depth and epochs.

**This notebook follows the pipeline's files and call signatures.** It writes raw activation
`.npy` files, reads them with `src.utils.load_all_data`, uses `_checkpoint_path` for SAE
filenames, calls `_evaluate_run_combined` to write its existing `.npz` format, and lets
`_save_concept_evo_pattern` and `compute_scores_for_reference_grid` reload those files.
The evaluator's matching logic, return value and artifact keys are unchanged.

This is a small teaching experiment, not a reproduction of ImageNet results. A smoke run
checks plumbing; meaningful reconstruction and classifier accuracy need more training.

## 0. Apply the small source edits before Run All

Only this notebook was edited. The exact source changes you need to apply are provided
in **Appendix A** at the end. They expose the activation root, log path and image-list
paths already assumed by the pipeline, infer the plot feature count from its saved
array, defer the optional WordNet import, propagate the requested CPU device and
support image previews on older Pillow versions.

**Do not apply the earlier proposed evaluator replacement or result-dictionary plotting
branch.** This version uses the current `_evaluate_run_combined(..., cache)` and
`_save_concept_evo_pattern(...)` interfaces. No changes to the evaluator or aggregation
module are required.

After those edits, restart the kernel. The first code cell checks the required integration
points before starting training. It does not patch or replace repository functions.

## 1. Environment and imports

Use an environment with PyTorch, torchvision, timm, NumPy, SciPy, pandas, h5py,
scikit-learn, matplotlib, seaborn, Pillow, PyYAML, tqdm and Jupyter/IPython.
Use a mutually compatible PyTorch/torchvision pair. The validation environment uses
Python 3.10, torch 1.13, torchvision 0.14, timm 1.0 and scikit-learn 1.0.

The repository provides activation collection, SAE models/training/checkpoints,
disk loading, combined evaluation, plotting, probing and migration scoring.
Notebook code handles the small Shapes3D sample, backbone training and file export.

In [ ]:
import ast
import contextlib
import inspect
import io
import json
import math
import os
import platform
import random
import sys
import tempfile
import time
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
import timm
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import yaml
from PIL import Image as PILImage
from IPython.display import display, Image as NotebookImage
from IPython.utils.capture import capture_output

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "base_sae.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside this repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils import load_all_data, feature_normalize

# Fail before training if the documented source edits have not been applied.
pending = []
if "activation_root" not in inspect.signature(load_all_data).parameters:
    pending.append("src/utils.py: activation_root")
tracking_tree = ast.parse((ROOT / "sae_track_method.py").read_text(encoding="utf-8"))
if any(isinstance(node, ast.ImportFrom) and node.module == "src.concept_type_classifier"
       for node in tracking_tree.body):
    pending.append("sae_track_method.py: remove the top-level WordNet import")
if pending:
    raise RuntimeError("Apply Appendix A and restart the kernel: " + "; ".join(pending))

import sae_track_method as tracking
import evo_aggregate_patterns as repo_metrics
from generate_acts_vit_fun_tiny import collect_hidden_states_timm
from src.sae_utils import (
    train_sae_fast, load_checkpoint, get_saes_similarities,
    concept_summary_stats, plot_metrics_figure,
)
from src.probing import _fit_logreg_probe

plot_source = inspect.getsource(tracking._save_concept_evo_pattern)
if not all(key in plot_source for key in ('args.get("log_path")', 'args.get("imglist_path")')):
    raise RuntimeError("Apply the plotting path edits in Appendix A before training.")
if 'activation_root' not in inspect.getsource(tracking._load_all_data_cached):
    raise RuntimeError("Forward activation_root in _load_all_data_cached; see Appendix A.")
print("Repository:", ROOT)
print("Python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)


c:\Users\rooben\anaconda3\envs\openood2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\rooben\anaconda3\envs\openood2\lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


## 2. Speed controls

Start with `PROFILE = "smoke"` to check the complete pipeline. The default `demo` keeps
five transformer blocks. Reduce `depth`, `embed_dim`, training samples or epochs for a
shorter run; `embed_dim` must remain divisible by `num_heads`.

The all-reference analysis compares every reference with every target, so reducing the
checkpoint grid helps substantially. Set `all_reference_analysis=False` to score only
the final reference, while still evaluating it across all saved checkpoints and blocks.
Each Run All creates a fresh output directory; old artifacts are not reused silently.

In [ ]:
PROFILE = 'demo'                         # 'demo' or 'smoke'
CFG = dict(
    seed=42, depth=5, embed_dim=192, num_heads=3,
    image_size=64, batch_size=64, cpu_threads=min(4, os.cpu_count() or 1),
    n_train=2048, n_val=512, n_analysis=512,
    n_sae_train=1024, n_sae_val=256,
    vit_epochs=12, checkpoint_epochs=[0, 1, 3, 6, 12],
    vit_lr=5e-4, weight_decay=0.01,
    sae_epochs=40, sae_expansion=2, sae_k=16, sae_batch_size=256,
    sae_lr=0.003, sae_z_l2=0.05,
    all_reference_analysis=True, pattern_images=True,
)
if PROFILE == 'smoke':
    CFG.update(n_train=128, n_val=64, n_analysis=64,
               n_sae_train=128, n_sae_val=64,
               vit_epochs=1, checkpoint_epochs=[0, 1], sae_epochs=3)
elif PROFILE != 'demo':
    raise ValueError("PROFILE must be 'demo' or 'smoke'")

# Optional overrides: uncomment before Run All, for example.
# CFG.update(depth=3, embed_dim=96, vit_epochs=4,
#            checkpoint_epochs=[0, 1, 2, 4], sae_epochs=15)

assert CFG['depth'] >= 2 and CFG['embed_dim'] % CFG['num_heads'] == 0
assert CFG['image_size'] in (32, 64)
assert min(CFG['n_train'], CFG['n_val'], CFG['n_analysis']) >= 32
assert 1 <= CFG['n_sae_train'] <= CFG['n_train']
assert 1 <= CFG['n_sae_val'] <= CFG['n_val']
assert 0 < CFG['sae_k'] <= CFG['embed_dim'] * CFG['sae_expansion']
assert CFG['vit_epochs'] >= 1 and CFG['sae_epochs'] >= 1
EPOCHS = sorted(set([0, CFG['vit_epochs']] + [int(e) for e in CFG['checkpoint_epochs']
                  if 0 <= e <= CFG['vit_epochs']]))
LAYERS = list(range(CFG['depth']))
REF = (EPOCHS[-1], LAYERS[-1])
DEVICE = torch.device('cpu')              # Explicitly CPU, even on a GPU machine.
torch.set_num_threads(CFG['cpu_threads'])

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_all(CFG['seed'])
DATA_PATH = ROOT / 'data' / 'shapes3d' / '3dshapes.h5'
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Expected local Shapes3D file: {DATA_PATH}')
OUTPUT_ROOT = ROOT / 'demo_outputs' / 'shapes3d_cpu'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = Path(tempfile.mkdtemp(prefix=PROFILE + '_', dir=OUTPUT_ROOT))
for name in ('vit', 'activations', 'analysis', 'figures', 'logs', 'images', 'imglists'):
    (RUN_DIR / name).mkdir()
(RUN_DIR / 'config.json').write_text(json.dumps(dict(CFG, epochs=EPOCHS), indent=2))
VERSIONS = dict(python=platform.python_version(), torch=torch.__version__, timm=timm.__version__,
                numpy=np.__version__, scipy=scipy.__version__, h5py=h5py.__version__)
(RUN_DIR / 'versions.json').write_text(json.dumps(VERSIONS, indent=2))
TIMES = {}
NOTEBOOK_START = time.perf_counter()
print(json.dumps(CFG, indent=2))
print(f'{len(EPOCHS)} checkpoints × {len(LAYERS)} layers = {len(EPOCHS)*len(LAYERS)} SAEs')
print('Reference (epoch, layer):', REF, '| output:', RUN_DIR)


## 3. Load a reproducible, disjoint Shapes3D sample

Classification predicts the four-valued **shape** factor. Train, validation and analysis
images are disjoint. The pipeline calls the analysis split `test`; it is not used for
classifier or SAE fitting.

In [ ]:
t0 = time.perf_counter()
FACTOR_NAMES = ['floor_hue', 'wall_hue', 'object_hue', 'scale', 'shape', 'orientation']
with h5py.File(DATA_PATH, 'r') as f:
    print({name: (f[name].shape, str(f[name].dtype)) for name in ('images', 'labels')})
    raw_factors = f['labels'][:]
    assert raw_factors.ndim == 2 and raw_factors.shape[1] == 6
    assert f['images'].shape[1:] == (64, 64, 3)
factor_values, factor_ids = [], np.empty(raw_factors.shape, dtype=np.int64)
for j in range(6):
    values, ids = np.unique(raw_factors[:, j], return_inverse=True)
    factor_values.append(values)
    factor_ids[:, j] = ids
assert [len(v) for v in factor_values] == [10, 10, 10, 8, 4, 15]
shape_ids = factor_ids[:, 4]
n_total = CFG['n_train'] + CFG['n_val'] + CFG['n_analysis']
assert n_total < len(shape_ids)
pool, _ = train_test_split(np.arange(len(shape_ids)), train_size=n_total,
                          stratify=shape_ids, random_state=CFG['seed'])
train_idx, rest = train_test_split(pool, train_size=CFG['n_train'], stratify=shape_ids[pool],
                                 random_state=CFG['seed'] + 1)
val_idx, analysis_idx = train_test_split(rest, train_size=CFG['n_val'], stratify=shape_ids[rest],
                                       random_state=CFG['seed'] + 2)
SPLIT_IDS = dict(train=np.sort(train_idx), val=np.sort(val_idx), analysis=np.sort(analysis_idx))
joined = np.concatenate(list(SPLIT_IDS.values()))
assert len(np.unique(joined)) == n_total, 'The three splits must not overlap.'
np.savez(RUN_DIR / 'split_indices.npz', **SPLIT_IDS)
np.savez(RUN_DIR / 'factor_values.npz', **dict(zip(FACTOR_NAMES, factor_values)))

def read_selected_images(path, indices):
    indices = np.sort(np.asarray(indices))
    selected = np.empty((len(indices), 64, 64, 3), dtype=np.uint8)
    with h5py.File(path, 'r') as f:
        ds = f['images']
        stride = ds.chunks[0] if ds.chunks else 2048
        for start in range(0, ds.shape[0], stride):
            lo, hi = np.searchsorted(indices, [start, start + stride])
            if hi > lo:
                slab = ds[start:min(start + stride, ds.shape[0])]
                selected[lo:hi] = slab[indices[lo:hi] - start]
                del slab
            if (start // stride + 1) % 8 == 0:
                print(f'Read image slabs: {start // stride + 1}/{math.ceil(ds.shape[0]/stride)}')
    return selected

all_selected_ids = np.sort(joined)
selected_rgb = read_selected_images(DATA_PATH, all_selected_ids)
RGB, X, Y, FACTORS = {}, {}, {}, {}
for split, ids in SPLIT_IDS.items():
    RGB[split] = selected_rgb[np.searchsorted(all_selected_ids, ids)]
    images = torch.from_numpy(RGB[split].copy()).permute(0, 3, 1, 2).float() / 255.0
    if CFG['image_size'] != 64:
        images = torch.nn.functional.interpolate(images, size=CFG['image_size'], mode='area')
    X[split] = (images - 0.5) / 0.5
    Y[split] = torch.from_numpy(shape_ids[ids].copy()).long()
    FACTORS[split] = factor_ids[ids].copy()
del selected_rgb, raw_factors, images
TIMES['data_loading'] = time.perf_counter() - t0
display(pd.DataFrame({s: np.bincount(y.numpy(), minlength=4) for s, y in Y.items()})
        .rename_axis('shape ID'))
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for ax, i in zip(axes.flat, range(12)):
    ax.imshow(RGB['train'][i])
    ax.set_title(f"shape={Y['train'][i].item()}, hue={FACTORS['train'][i, 2]}")
    ax.axis('off')
fig.tight_layout()
fig.savefig(RUN_DIR / 'figures' / 'sample_images.png', dpi=120)
plt.show()
print(f"Data loading: {TIMES['data_loading']:.1f}s")


## 4. Train ViT and write the pipeline-style validation log

Save actual model states at the scheduled epochs, including random initialization at
epoch 0. The exported log has one `Val Acc` line per completed epoch, including epoch 0,
so the pipeline can index it using the saved checkpoint epoch numbers.

In [ ]:
def build_vit():
    return timm.create_model('vit_tiny_patch16_224', pretrained=False,
        img_size=CFG['image_size'], depth=CFG['depth'], embed_dim=CFG['embed_dim'],
        num_heads=CFG['num_heads'], num_classes=4, drop_path_rate=0.0).to(DEVICE)

seed_all(CFG['seed'])
model = build_vit()
assert len(model.blocks) == CFG['depth'] and model.embed_dim == CFG['embed_dim']
assert model.blocks[0].attn.num_heads == CFG['num_heads']
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Tokens per image: {model.patch_embed.num_patches} patches + 1 CLS')
print('All parameters on:', {p.device.type for p in model.parameters()})

def loader_for(split, shuffle=False):
    return DataLoader(TensorDataset(X[split], Y[split]), batch_size=CFG['batch_size'],
                      shuffle=shuffle, num_workers=0, pin_memory=False)

@torch.no_grad()
def evaluate_classifier(net, split):
    net.eval()
    loss_sum, correct, count = 0.0, 0, 0
    for xb, yb in loader_for(split):
        logits = net(xb)
        loss_sum += nn.functional.cross_entropy(logits, yb, reduction='sum').item()
        correct += (logits.argmax(1) == yb).sum().item()
        count += len(yb)
    return loss_sum / count, correct / count

def save_vit(epoch):
    torch.save(dict(model=model.state_dict(), epoch=epoch),
               RUN_DIR / 'vit' / f'epoch_{epoch:03d}.pt')

def read_own_checkpoint(path):
    # These files are created by this notebook. The fallback supports older torch.
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(path, map_location='cpu')

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['vit_lr'], weight_decay=CFG['weight_decay'])
save_vit(0)
initial_loss, initial_acc = evaluate_classifier(model, 'val')
history = [dict(epoch=0, train_loss=np.nan, train_acc=np.nan,
                val_loss=initial_loss, val_acc=initial_acc, seconds=0.0)]
t0 = time.perf_counter()
for epoch in range(1, CFG['vit_epochs'] + 1):
    epoch_start = time.perf_counter()
    model.train()
    loss_sum, correct, count = 0.0, 0, 0
    for step, (xb, yb) in enumerate(loader_for('train', shuffle=True)):
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = nn.functional.cross_entropy(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        loss_sum += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        count += len(yb)
        if epoch == 1 and step == min(2, math.ceil(len(Y['train']) / CFG['batch_size']) - 1):
            rate = (time.perf_counter() - epoch_start) / (step + 1)
            estimate = rate * math.ceil(len(Y['train']) / CFG['batch_size']) * CFG['vit_epochs']
            print(f'Observed {rate:.3f}s/batch; rough backbone training estimate {estimate/60:.1f} min')
    val_loss, val_acc = evaluate_classifier(model, 'val')
    elapsed = time.perf_counter() - epoch_start
    history.append(dict(epoch=epoch, train_loss=loss_sum/count, train_acc=correct/count,
                        val_loss=val_loss, val_acc=val_acc, seconds=elapsed))
    if epoch in EPOCHS:
        save_vit(epoch)
    print(f'epoch {epoch:2d} | train loss {loss_sum/count:.3f} | val accuracy {val_acc:.3f} | {elapsed:.1f}s')
TIMES['vit_training'] = time.perf_counter() - t0
history_df = pd.DataFrame(history)
history_df.to_csv(RUN_DIR / 'training_history.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
history_df.plot(x='epoch', y=['train_loss', 'val_loss'], ax=axes[0])
history_df.plot(x='epoch', y=['train_acc', 'val_acc'], ax=axes[1])
axes[1].axhline(0.25, color='gray', ls='--', label='balanced chance')
axes[1].set_ylim(0, 1)
axes[1].legend()
fig.tight_layout()
fig.savefig(RUN_DIR / 'figures' / 'classifier_training.png', dpi=120)
plt.show()

LOG_PATH = RUN_DIR / "logs" / "log.txt"
LOG_PATH.write_text(
    "".join(f"Epoch {int(row.epoch)} Val Acc {row.val_acc:.10f}\n"
            for row in history_df.itertuples()),
    encoding="utf-8",
)


## 5. Use the pipeline's experiment paths

`ind_name="shapes3d_cpu"` is an experiment tag for the same Shapes3D images. It selects
the loader's existing numeric-label format (`*_y.npy`), rather than its special
`ind_name="shapes3d"` filename-to-factor-dictionary parser. Thus the evaluator receives
the shape-class label vector it already expects.

The checkpoint directory deliberately follows the pipeline's parent-directory
convention. `vit_size` is its legacy MLP-width setting, so it is `4 * embed_dim`.
Nothing in the notebook constructs an alternative SAE checkpoint filename.

In [ ]:
DATASET_TAG = "shapes3d_cpu"
SEED_TAG = f"_s{CFG['seed']}"
EXPERIMENT_TAG = f"vit_{DATASET_TAG}{SEED_TAG}"
ACTIVATION_ROOT = RUN_DIR / "activations"
SAE_EXPERIMENT_DIR = RUN_DIR / "sae_checkpoints" / EXPERIMENT_TAG
EVO_DIR = SAE_EXPERIMENT_DIR / "evo_patterns"
IMAGE_DIR = RUN_DIR / "images"
IMGLIST_PATH = RUN_DIR / "imglists" / f"test_{DATASET_TAG}.txt"

BASE_ARGS = dict(
    backbone="vit", ind_name=DATASET_TAG, seed=SEED_TAG, step=0,
    activation_root=str(ACTIVATION_ROOT),
    train_device="cpu", eval_device="cpu",
    train_split="train", eval_split="test", keywords=["train", "test"],
    norm=True, trained_on="cls", train_patches=False, test_patches=False,
    fixed_sae=False, sae_runs=1, skip_existing=False, cache_activations=False,
    run_seed_base=CFG["seed"] + 1000,
    sae_type="batch_top_k_sae", tied=False, circuits=True,
    expansion_coef=CFG["sae_expansion"], k=CFG["sae_k"],
    sae_epochs=CFG["sae_epochs"], batch_size=CFG["sae_batch_size"],
    lr=CFG["sae_lr"], l1_lambda=0.0, l2_lambda=CFG["sae_z_l2"],
    verbose_every=0, vit_size=4 * CFG["embed_dim"], num_heads=CFG["num_heads"],
    num_classes=4, img_size=CFG["image_size"], total_vit_epochs=CFG["vit_epochs"],
    pretrained_vit_IN1K=False, log_path=str(LOG_PATH),
    data_dir=str(IMAGE_DIR), imglist_path=str(IMGLIST_PATH),
    align_method="spearman", align_min_sim=0.05,
    align_compute_stability=False, align_compute_chance=False,
    topk_k=min(20, CFG["n_analysis"]), topk_metric="jaccard",
    topk_threshold=None, topk_act_threshold=0.0, topk_ranking="cls",
    combined=True, combined_images=True, concept_types=False, tag="_cls_only",
)

def args_for_reference(reference):
    epoch, layer = reference
    name = f"refepoch_{epoch}_reflayer_{layer}"
    checkpoint_folder = SAE_EXPERIMENT_DIR / name
    return dict(
        BASE_ARGS, ref_epoch=epoch, ref_layer=layer,
        checkpoint_folder=str(checkpoint_folder),
        results_dir=str(checkpoint_folder / "eval_results"),
        plots_dir=str(RUN_DIR / "figures" / EXPERIMENT_TAG / name),
    )

PIPELINE_ARGS = args_for_reference(REF)
(RUN_DIR / "pipeline_config.yaml").write_text(
    yaml.safe_dump(PIPELINE_ARGS, sort_keys=False), encoding="utf-8"
)
assert Path(PIPELINE_ARGS["results_dir"]).parent.parent / "evo_patterns" == EVO_DIR
print("Activation store:", ACTIVATION_ROOT / EXPERIMENT_TAG)
print("SAE checkpoint example:", tracking._checkpoint_path(PIPELINE_ARGS, *REF, 0))
print("Pipeline evolution files:", EVO_DIR)


## 6. Export activation arrays and an ordered image list

For each saved backbone, write:

```text
activations/vit_shapes3d_cpu_s42/epoch0_step0/
    vit_shapes3d_cpu_train.npy         # raw [SAE training images, blocks, width]
    vit_shapes3d_cpu_train_y.npy       # shape labels
    vit_shapes3d_cpu_train_logits.npy
    vit_shapes3d_cpu_train_ids.npy     # extra row-identity sidecar
    vit_shapes3d_cpu_train_factors.npy # extra six-factor sidecar
    ... val and test files ...
```

The `.npy` activation layout, split suffixes and label/logit filenames match
`load_all_data`. Save **raw** representations: normalization belongs to the repository
loader. The loader selects a layer on reading. Sidecars verify sample order and supply
the factor probes; they do not change the loader's return format.

Export PNGs and a `<relative_filename> <class_id>` image list in the exact same order
as `test.npy`. This lets the existing pattern plotter load Shapes3D images from disk.

In [ ]:
t0 = time.perf_counter()
rng = np.random.default_rng(CFG["seed"] + 3)
SAE_FIT_ROWS = np.sort(rng.choice(len(Y["train"]), CFG["n_sae_train"], replace=False))
SAE_VAL_ROWS = np.sort(rng.choice(len(Y["val"]), CFG["n_sae_val"], replace=False))
np.savez(RUN_DIR / "sae_fit_rows.npz", train=SAE_FIT_ROWS, val=SAE_VAL_ROWS)
EXPORT_SPLITS = {
    "train": ("train", SAE_FIT_ROWS),
    "val": ("val", SAE_VAL_ROWS),
    "test": ("analysis", np.arange(len(Y["analysis"]))),
}
EXPORT_IDS = {
    split: SPLIT_IDS[source][rows]
    for split, (source, rows) in EXPORT_SPLITS.items()
}

def activation_prefix(epoch, split):
    return (ACTIVATION_ROOT / EXPERIMENT_TAG / f"epoch{epoch}_step0"
            / f"vit_{DATASET_TAG}_{split}")

for epoch in EPOCHS:
    model.load_state_dict(
        read_own_checkpoint(RUN_DIR / "vit" / f"epoch_{epoch:03d}.pt")["model"],
        strict=True,
    )
    for split, (source, rows) in EXPORT_SPLITS.items():
        loader = DataLoader(TensorDataset(X[source][rows]), batch_size=CFG["batch_size"],
                            shuffle=False, num_workers=0)
        cls_chunks, logit_chunks = [], []
        for (xb,) in loader:
            cls, patches, logits = collect_hidden_states_timm(model, xb, need_patches=False)
            assert patches is None
            cls_chunks.append(cls.cpu())
            logit_chunks.append(logits.cpu())
        raw = torch.cat(cls_chunks).numpy().astype(np.float32)
        logits_np = torch.cat(logit_chunks).numpy()
        assert raw.shape == (len(rows), CFG["depth"], CFG["embed_dim"])
        prefix = activation_prefix(epoch, split)
        prefix.parent.mkdir(parents=True, exist_ok=True)
        np.save(str(prefix) + ".npy", raw)
        np.save(str(prefix) + "_y.npy", Y[source][rows].numpy())
        np.save(str(prefix) + "_logits.npy", logits_np)
        np.save(str(prefix) + "_ids.npy", EXPORT_IDS[split])
        np.save(str(prefix) + "_factors.npy", FACTORS[source][rows])
    print("Exported checkpoint", epoch)

image_lines = []
for row, image_id in enumerate(EXPORT_IDS["test"]):
    filename = f"shape_{int(image_id):06d}.png"
    PILImage.fromarray(RGB["analysis"][row]).save(IMAGE_DIR / filename)
    image_lines.append(f"{filename} {int(Y['analysis'][row])}\n")
IMGLIST_PATH.write_text("".join(image_lines), encoding="utf-8")

def load_disk_data(epoch, layer, splits=("train", "test")):
    # All arrays are loaded by the repository, with its normalization behavior.
    return load_all_data(
        epoch, BASE_ARGS["step"], BASE_ARGS["seed"], BASE_ARGS["norm"], list(splits),
        layer_nr=layer, backbone=BASE_ARGS["backbone"], ind_name=DATASET_TAG,
        train_patches=False, test_patches=False,
        activation_root=BASE_ARGS["activation_root"],
    )

# Verify sample identity/order at every point before matching.
for epoch in EPOCHS:
    for split, (source, rows) in EXPORT_SPLITS.items():
        prefix = activation_prefix(epoch, split)
        np.testing.assert_array_equal(np.load(str(prefix) + "_ids.npy"), EXPORT_IDS[split])
        np.testing.assert_array_equal(np.load(str(prefix) + "_y.npy"), Y[source][rows].numpy())
sample = load_disk_data(EPOCHS[0], LAYERS[0])
np.testing.assert_allclose(
    np.linalg.norm(sample["test"]["activations"], axis=1), 1.0, atol=2e-6
)
TIMES["activation_export"] = time.perf_counter() - t0
del cls_chunks, logit_chunks, raw, logits_np, cls, logits, sample
print("Exported", len(image_lines), "ordered test images and matching image-list rows.")


## 7. Train the pipeline's SAE grid and reload its checkpoints

`train_sae_grid` reads the exported files through `_load_all_data_cached` and writes
the exact names returned by `_checkpoint_path`. Use the existing `train_sae_fast`
trainer for all grid points to avoid the other trainer's 16-worker DataLoader on CPU.
The pipeline's common initialization seed per run is retained; each grid point still
gets a separate SAE fitted on that point's activations.

Reconstruction uses `sae(batch)` and its gated code. Tracking uses `sae.encode(x)`,
which currently returns the ungated ReLU readout for BatchTopK. Reconstruction metrics
use a fixed validation batch size because BatchTopK gating depends on the batch.

In [ ]:
def reconstruction_metrics(sae, x):
    sae.eval()
    squared_error, active_count = 0.0, 0
    ever_active = torch.zeros(sae.encoder.out_features, dtype=torch.bool)
    with torch.no_grad():
        for batch in x.split(CFG['sae_batch_size']):
            recon, sparse_z = sae(batch)
            squared_error += (recon - batch).square().sum().item()
            active_count += (sparse_z > 0).sum().item()
            ever_active |= (sparse_z > 0).any(0)
    mean_baseline_error = (x - x.mean(0)).square().sum().item()
    return dict(mse=squared_error/x.numel(),
                fvu=squared_error/max(mean_baseline_error, 1e-12),
                mean_l0=active_count/len(x),
                inactive_fraction=(~ever_active).float().mean().item())

t0 = time.perf_counter()
with capture_output() as captured:
    sae_paths = tracking.train_sae_grid(
        PIPELINE_ARGS, EPOCHS, LAYERS, *REF,
        load_data=lambda epoch, layer: tracking._load_all_data_cached(
            epoch, layer, PIPELINE_ARGS, None
        ),
        checkpoint_path=lambda epoch, layer, run_idx: tracking._checkpoint_path(
            PIPELINE_ARGS, epoch, layer, run_idx
        ),
        trainer=train_sae_fast,
    )
(RUN_DIR / "logs" / "sae_training.log").write_text(
    captured.stdout + captured.stderr, encoding="utf-8"
)

SAES, USED_IN_FIT, quality_records = {}, {}, []
for epoch in EPOCHS:
    for layer in LAYERS:
        with contextlib.redirect_stdout(io.StringIO()):
            data = load_disk_data(epoch, layer, ("train", "val"))
        fit_x = torch.from_numpy(data["train"]["activations"])
        val_x = torch.from_numpy(data["val"]["activations"])
        path = tracking._checkpoint_path(PIPELINE_ARGS, epoch, layer, 0)
        assert path == sae_paths[(0, epoch, layer)] and path.is_file()
        sae = load_checkpoint(path, device="cpu").eval()
        SAES[(epoch, layer)] = sae
        metrics = reconstruction_metrics(sae, val_x)
        used = torch.zeros(sae.encoder.out_features, dtype=torch.bool)
        with torch.no_grad():
            for batch in fit_x.split(CFG["sae_batch_size"]):
                _, sparse_z = sae(batch)
                used |= (sparse_z > 0).any(0)
        USED_IN_FIT[(epoch, layer)] = used.numpy()
        quality_records.append(dict(epoch=epoch, layer=layer, **metrics))
        print(f"SAE e={epoch}, l={layer}: validation FVU {metrics['fvu']:.3f}, L0 {metrics['mean_l0']:.1f}")
quality_df = pd.DataFrame(quality_records)
quality_df.to_csv(RUN_DIR / "analysis" / "sae_quality.csv", index=False)
np.savez(RUN_DIR / "analysis" / "selected_in_fit.npz",
         **{f"e{e}_l{l}": mask for (e, l), mask in USED_IN_FIT.items()})
TIMES["sae_fitting"] = time.perf_counter() - t0
display(quality_df.groupby("epoch")[["mse", "fvu", "mean_l0", "inactive_fraction"]].mean())
print("FVU < 1 beats a validation-mean baseline; it does not establish interpretability.")


## 8. Run combined evaluation, then reload its file

Call `_evaluate_run_combined` with the same arguments as the experiment script.
It loads activations and checkpoints itself and writes:

```text
sae_checkpoints/vit_shapes3d_cpu_s42/evo_patterns/
    evolution_patterns_e12_l4.npz
```

The saved keys are `cor_res`, `top_res`, `concepts`, `epochs`, `cka`, and `pre_loss`.
For each source feature, combined matching takes the maximum of its best match in an
independently trained target SAE and its corresponding fixed-SAE readout. All target
features participate, as in the pipeline. We do not apply the old notebook's additional
matching masks, or recompute separate independent/fixed/shuffled arrays.

`pre_loss` is reconstruction by the **reference SAE** on each target's analysis
activations, not the independent SAE validation loss above.

In [ ]:
t0 = time.perf_counter()

def run_pipeline_reference(reference):
    args = args_for_reference(reference)
    config_dir = Path(args["results_dir"]) / "run_0"
    config_dir.mkdir(parents=True, exist_ok=True)
    (config_dir / "config.yaml").write_text(yaml.safe_dump(args, sort_keys=False), encoding="utf-8")
    with capture_output() as captured:
        tracking._evaluate_run_combined(0, args, EPOCHS, LAYERS, *reference, None)
    plt.close("all")
    log = RUN_DIR / "logs" / f"evaluate_e{reference[0]}_l{reference[1]}.log"
    log.write_text(captured.stdout + captured.stderr, encoding="utf-8")
    path = EVO_DIR / f"evolution_patterns_e{reference[0]}_l{reference[1]}.npz"
    if not path.is_file():
        raise RuntimeError(f"The pipeline did not write {path}")
    return path

reference_path = run_pipeline_reference(REF)
with np.load(reference_path, allow_pickle=False) as stored:
    evolution = {key: stored[key].copy() for key in stored.files}
assert set(evolution) == {"cor_res", "top_res", "concepts", "epochs", "cka", "pre_loss"}
np.testing.assert_array_equal(evolution["epochs"], EPOCHS)
assert evolution["cor_res"].shape == (
    CFG["embed_dim"] * CFG["sae_expansion"], len(LAYERS), len(EPOCHS)
)
with contextlib.redirect_stdout(io.StringIO()):
    ref_data = load_disk_data(*REF, splits=("test",))
ref_x = torch.from_numpy(ref_data["test"]["activations"])
ref_codes = SAES[REF].encode(ref_x).cpu()
nonconstant = ref_codes.std(0, unbiased=False).numpy() > 1e-8
self_li, self_ei = LAYERS.index(REF[1]), EPOCHS.index(REF[0])
np.testing.assert_allclose(
    evolution["cor_res"][nonconstant, self_li, self_ei], 1.0, atol=2e-5
)

# Check the repository's Spearman implementation on a tied-rank example.
test_a = torch.tensor([[0.], [0.], [1.], [2.], [3.]])
test_b = torch.tensor([[0.], [2.], [1.], [0.], [3.]])
rho = get_saes_similarities(
    test_a, test_b, method="spearman", metric="jaccard", device="cpu", k=3
)["cor_sim"][0, 0]
np.testing.assert_allclose(rho, spearmanr(test_a[:, 0], test_b[:, 0]).statistic, atol=2e-5)
TIMES["reference_evaluation"] = time.perf_counter() - t0
print("Reloaded pipeline result:", reference_path)
print({key: value.shape for key, value in evolution.items()})


## 9. Let the pipeline reload and plot the saved patterns

The plotter reads the same `.npz`, checkpoint, activations, `Val Acc` log, PNG files
and image list that the script uses. It keeps its existing first-61-feature limit and
activation/entropy filters. Some small smoke runs may produce only the loss/CKA
report; no synthetic pattern is substituted when a feature fails those filters.

`pattern_images=False` skips the extra per-class galleries. The pipeline's pattern
heatmaps still include their low/mid/high image strip. Only a few saved figures are
displayed inline; the rest remain on disk.

In [ ]:
t0 = time.perf_counter()
with capture_output() as captured:
    tracking._save_concept_evo_pattern(
        0, PIPELINE_ARGS, EPOCHS, LAYERS, *REF,
        method="max", want_images=CFG["pattern_images"], eval_device="cpu",
    )
plt.close("all")
(RUN_DIR / "logs" / "pattern_plots.log").write_text(
    captured.stdout + captured.stderr, encoding="utf-8"
)
PATTERN_DIR = Path(PIPELINE_ARGS["plots_dir"]) / "run_0" / f"combined_{PIPELINE_ARGS['tag']}"
loss_figure = PATTERN_DIR / f"losses_ckas_ref_e{REF[0]}_l{REF[1]}.png"
pattern_paths = sorted(PATTERN_DIR.glob(f"max_run_0_ref_e{REF[0]}_l{REF[1]}_concept*.png"))
if loss_figure.is_file():
    display(NotebookImage(filename=str(loss_figure)))
else:
    raise RuntimeError(f"Missing loss/CKA plot. Read {RUN_DIR / 'logs' / 'pattern_plots.log'}")
for path in pattern_paths[:2]:
    display(NotebookImage(filename=str(path)))
print(f"Saved {len(pattern_paths)} feature-pattern reports in {PATTERN_DIR}")
TIMES["pattern_plots"] = time.perf_counter() - t0


## 10. Inspect a feature and probe the known factors

Select a nonconstant readout for the example below. This choice affects only this
illustration, not the pipeline's matching or scoring. The training-use mask remains a
diagnostic. Factor probes train on the exported SAE training subset and evaluate on
the exported test images, using the IDs/factor sidecars to verify alignment.

In [ ]:
candidate_ids = np.flatnonzero(nonconstant & USED_IN_FIT[REF])
if len(candidate_ids) == 0:
    candidate_ids = np.flatnonzero(nonconstant)
if len(candidate_ids) == 0:
    raise RuntimeError("No varying readout to illustrate; increase data or SAE epochs.")
feature_id = int(candidate_ids[0])
stats_df = pd.DataFrame(
    evolution["concepts"], columns=["sparsity", "mean_activation", "label_entropy"]
).rename_axis("feature_id")
stats_df["nonconstant"] = nonconstant
stats_df["used_in_fit"] = USED_IN_FIT[REF]
stats_df.to_csv(RUN_DIR / "analysis" / "reference_feature_stats.csv")
display(stats_df.loc[candidate_ids[:8]])

fig, axes = plt.subplots(1, 3, figsize=(12, 3), constrained_layout=True)
for ax, key, title, limits in zip(
    axes, ["cor_res", "top_res", "cka"],
    ["Combined Spearman", "Combined top-k Jaccard", "Representation CKA"],
    [(-1, 1), (0, 1), (0, 1)],
):
    values = evolution[key][feature_id] if key != "cka" else evolution[key]
    im = ax.imshow(values, origin="lower", aspect="auto", vmin=limits[0], vmax=limits[1])
    ax.set_xticks(range(len(EPOCHS)), EPOCHS)
    ax.set_yticks(range(len(LAYERS)), LAYERS)
    ax.set(xlabel="Completed epoch", ylabel="Block", title=title)
    fig.colorbar(im, ax=ax)
fig.suptitle(f"Source feature {feature_id}: values read from the pipeline artifact")
fig.savefig(RUN_DIR / "figures" / "saved_pattern_overview.png", dpi=130)
plt.show()

top_values, top_rows = torch.topk(ref_codes[:, feature_id], k=min(8, len(ref_codes)))
image_entries = [line.rsplit(" ", 1) for line in IMGLIST_PATH.read_text().splitlines()]
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, row, value in zip(axes.flat, top_rows.tolist(), top_values.tolist()):
    rel_path, label = image_entries[row]
    with PILImage.open(IMAGE_DIR / rel_path) as image:
        ax.imshow(np.asarray(image))
    ax.set_title(f"ID {EXPORT_IDS['test'][row]}\nshape {label}, readout {value:.2f}")
    ax.axis("off")
fig.suptitle(f"Top exported images for feature {feature_id}")
fig.tight_layout()
fig.savefig(RUN_DIR / "figures" / "top_activating_images.png", dpi=130)
plt.show()

# Extraction ended at the final checkpoint, so model is the final classifier.
analysis_loss, analysis_accuracy = evaluate_classifier(model, "analysis")
(RUN_DIR / "analysis" / "classifier_evaluation.json").write_text(
    json.dumps(dict(loss=analysis_loss, accuracy=analysis_accuracy), indent=2)
)
print(f"Classifier held-out accuracy: {analysis_accuracy:.3f}; balanced chance: 0.25")
train_factors = np.load(str(activation_prefix(REF[0], "train")) + "_factors.npy")
test_factors = np.load(str(activation_prefix(REF[0], "test")) + "_factors.npy")
probe_records = []
for layer in LAYERS:
    with contextlib.redirect_stdout(io.StringIO()):
        data = load_disk_data(REF[0], layer)
    for j, name in enumerate(FACTOR_NAMES):
        score = _fit_logreg_probe(
            data["train"]["activations"], train_factors[:, j],
            data["test"]["activations"], test_factors[:, j], max_iter=300,
        )
        probe_records.append(dict(layer=layer, factor=name, balanced_accuracy=score,
                                  chance=1 / len(factor_values[j])))
probe_df = pd.DataFrame(probe_records)
probe_df.to_csv(RUN_DIR / "analysis" / "factor_probes.csv", index=False)
display(probe_df.pivot(index="layer", columns="factor", values="balanced_accuracy").round(3))


## 11. Generate and aggregate the pipeline's reference files

Evaluate each requested reference using the same saved SAE grid. Then call
`compute_scores_for_reference_grid` with the **directory of pipeline `.npz` files**.
That function reloads `cor_res`, `concepts` and `epochs`, applies its existing
`sparsity > 0.001` inclusion rule, and calls the repository's concept/aggregate scorers.
Here `sparsity` means fraction of positive readouts.

The short grid uses two-checkpoint onset/disappearance windows and smaller
layer-distance thresholds than the 12-layer ImageNet experiment. These are explicit
analysis parameters, not changes to the scoring implementation. Smoke-run classes
should not be interpreted as evidence of learned semantic migration.

In [ ]:
t0 = time.perf_counter()
REFERENCE_EPOCHS = EPOCHS if CFG["all_reference_analysis"] else [REF[0]]
REFERENCE_LAYERS = LAYERS if CFG["all_reference_analysis"] else [REF[1]]
for epoch in REFERENCE_EPOCHS:
    for layer in REFERENCE_LAYERS:
        if (epoch, layer) != REF:
            run_pipeline_reference((epoch, layer))
        print("Pipeline result ready:", (epoch, layer))

SCORE_KWARGS = dict(
    input_layout="NLT", onset_k=2, disappearance_k=2,
    start_end_fraction=0.25, drift_threshold_layers=0.5,
    min_directional_efficiency=0.35, min_layer_localization=0.15,
    max_width=max(1.0, (CFG["depth"] - 1) * 0.6), choice="com",
)
(RUN_DIR / "analysis" / "score_config.json").write_text(
    json.dumps(SCORE_KWARGS, indent=2)
)
with capture_output() as captured:
    all_scores_df, summary_df = repo_metrics.compute_scores_for_reference_grid(
        similarity_source=str(EVO_DIR),
        reference_epochs=REFERENCE_EPOCHS, reference_layers=REFERENCE_LAYERS,
        layer_values=LAYERS, denominator="all", **SCORE_KWARGS,
    )
(RUN_DIR / "logs" / "aggregation.log").write_text(
    captured.stdout + captured.stderr, encoding="utf-8"
)
all_scores_df.to_csv(RUN_DIR / "analysis" / "all_reference_feature_metrics.csv", index=False)
summary_df.to_csv(RUN_DIR / "analysis" / "all_reference_summary.csv", index=False)
feature_scores = all_scores_df[
    (all_scores_df["reference_epoch"] == REF[0])
    & (all_scores_df["reference_layer"] == REF[1])
].copy()
feature_scores.to_csv(RUN_DIR / "analysis" / "reference_migration_metrics.csv", index=False)
display(feature_scores[[
    "concept_id", "has_emerged", "emergence_epoch",
    "signed_mean_drift_active", "mean_layer_width", "drift_class",
]].head(12))
figures = repo_metrics.plot_aggregate_heatmaps(
    summary_df,
    metrics=["earlier_migratory_pct", "deeper_migratory_pct", "mean_layer_localization"],
    reference_epochs=REFERENCE_EPOCHS, reference_layers=REFERENCE_LAYERS,
    save_path=str(RUN_DIR / "figures" / "aggregate"),
    dpi=130, close_after_save=True, datasettag="Shapes3D CPU",
)
for figure in figures.values():
    display(figure)
display(summary_df[["reference_epoch", "reference_layer", "n_concepts",
                    "n_emerged", "earlier_migratory_pct", "deeper_migratory_pct"]])
TIMES["all_reference_analysis"] = time.perf_counter() - t0


## 12. Check the artifacts and preserve the experiment

Checks cover disjoint sampling, ordered activation/image rows, changed backbone
weights, SAE dimensions and sparsity, finite saved similarities, reference
self-matches for varying readouts, and the aggregator's feature inclusion rule.

The notebook does not alter matching to exclude inactive units. Interpretation should
consider the reconstruction diagnostics and feature activation frequencies alongside
the pipeline's migration classes. Its saved combined similarities do not include a
shuffled baseline, so the earlier notebook's shuffled-control claim is not made here.

In [ ]:
assert len(np.unique(joined)) == len(joined)
assert len(SAES) == len(EPOCHS) * len(LAYERS)
initial_state = read_own_checkpoint(RUN_DIR / "vit" / "epoch_000.pt")["model"]
final_state = read_own_checkpoint(RUN_DIR / "vit" / f"epoch_{EPOCHS[-1]:03d}.pt")["model"]
assert not torch.equal(initial_state["head.weight"], final_state["head.weight"])
assert all(sae.encoder.in_features == CFG["embed_dim"] for sae in SAES.values())
assert quality_df["mean_l0"].max() <= CFG["sae_k"] + 1e-5
assert len(image_entries) == len(EXPORT_IDS["test"])
assert all((IMAGE_DIR / entry[0]).is_file() for entry in image_entries)
np.testing.assert_array_equal(
    [int(entry[1]) for entry in image_entries],
    np.load(str(activation_prefix(REF[0], "test")) + "_y.npy"),
)
for epoch in REFERENCE_EPOCHS:
    for layer in REFERENCE_LAYERS:
        with np.load(EVO_DIR / f"evolution_patterns_e{epoch}_l{layer}.npz") as result:
            assert result["cor_res"].shape == evolution["cor_res"].shape
            assert np.isfinite(result["cor_res"]).all()
            assert np.isfinite(result["top_res"]).all()
            assert np.isfinite(result["pre_loss"]).all()
            assert np.abs(result["cor_res"]).max() <= 1 + 2e-5
            np.testing.assert_array_equal(result["epochs"], EPOCHS)
expected_ids = np.flatnonzero(evolution["concepts"][:, 0] > 0.001)
assert set(feature_scores["concept_id"]) == set(expected_ids)
assert len(summary_df) == len(REFERENCE_EPOCHS) * len(REFERENCE_LAYERS)
TIMES["total_wall"] = time.perf_counter() - NOTEBOOK_START
(RUN_DIR / "timings.json").write_text(json.dumps(TIMES, indent=2))
display(pd.Series(TIMES, name="seconds").round(2).to_frame())
display(pd.DataFrame([
    ("vit/", "Random initialization and scheduled trained backbone checkpoints"),
    ("activations/<experiment>/epochE_step0/", "Pipeline .npy activations, labels, logits and identity/factor sidecars"),
    ("sae_checkpoints/<experiment>/*.pt", "Checkpoints named by _checkpoint_path"),
    ("sae_checkpoints/<experiment>/evo_patterns/", "Unchanged combined evaluator .npz outputs"),
    ("images/ + imglists/", "Exported test PNGs and ordered pipeline image list"),
    ("logs/", "Val Acc log and captured pipeline execution logs"),
    ("analysis/", "SAE quality, factor probes, scorer settings and migration CSVs"),
    ("figures/", "Repository pattern reports, class galleries and aggregate heatmaps"),
], columns=["artifact", "contents"]))
print("All checks passed. Outputs:", RUN_DIR)


## Appendix B. Rerunning and interpreting a small experiment

Use the saved `pipeline_config.yaml`, `config.json`, `score_config.json`, split IDs and
package versions to identify a run. `cache_activations=False` keeps the loader's full
dictionary and evaluation-only paths explicit. The original script's `main()` builds a
large windowed epoch schedule, so this notebook calls its stage functions directly with
the checkpoint grid it actually exported.

Increasing `depth`, `embed_dim`, the number of checkpoint epochs, or SAE expansion
increases both training and matching work. Start by inspecting classifier accuracy and
SAE validation FVU; then increase data/epochs and repeat seeds before drawing claims
about feature migration. The HDF5 source is opened read-only, and all outputs go into
the new run directory.